### IMPORTS:

In [46]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score

In [47]:
df = pd.read_excel("Online Retail.xlsx")
print(f"Shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst rows:\n{df.head()}")

Shape: (541909, 8)

Column types:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

Missing values:
InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

First rows:
  InvoiceNo StockCode                          Description  Quantity  \
0    536365    85123A   WHITE HANGING HEART T-LIGHT HOLDER         6   
1    536365     71053                  WHITE METAL LANTERN         6   
2    536365    84406B       CREAM CUPID HEARTS COAT HANGER         8   
3    536365    84029G  KNITTED UNION FLAG HOT WATER BOTTLE         6   
4    536365    84029E       RED WOOLLY HOTTIE WHITE HEART.         6   

          InvoiceDate  UnitPrice  CustomerID         Country

## Task 1: Data Profiling and Missing Values 

### 1.1 — Profile the raw data

In [48]:
print(f"Number of rows, columns, and memory usage:")
df.info()
print(f"Missing value counts for each column: {df.isnull().sum()}")
print(f"Missing value percentages for each column: {((df.isnull().sum()/len(df))*100).round(2)}")
print(f"Number of unique values per column: {df.nunique()}")
print("Basic statistics for numeric columns (min, max, mean, median, std):")
df.describe().T

Number of rows, columns, and memory usage:
<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB
Missing value counts for each column: InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64
Missing value percentages for each column

,count,mean,min,25%,50%,75%,max,std
Quantity,541909.0,9.55225,-80995.0,1.0,3.0,10.0,80995.0,218.081158
InvoiceDate,541909,2011-07-04 13:34:57.156386,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,541909.0,4.611114,-11062.06,1.25,2.08,4.13,38970.0,96.759853
CustomerID,406829.0,15287.69057,12346.0,13953.0,15152.0,16791.0,18287.0,1713.600303


### 1.2 — Classify the missing data types

In [49]:
df["has_id"]=df["CustomerID"].notnull()
analysis_id=df.groupby("has_id").agg({
    "Quantity": ["mean", "median"],
    "UnitPrice": ["mean", "median"],
    "Country": lambda x: x.mode()[0]
})
print(f"Comparison between rows with and without missing values:")
print(analysis_id)
missing_desc=df[df["Description"].isnull()]
unique_codes_missing_desc=missing_desc['StockCode'].unique()
print(f"Number of rows with missing Description: {len(missing_desc)}")
print(f"Number of unique StockCodes with missing descriptions: {len(unique_codes_missing_desc)}")
if len(unique_codes_missing_desc) > 0:
    sample_code = unique_codes_missing_desc[0]
    recovered = df[df['StockCode'] == sample_code]['Description'].dropna().unique()
    print(f"\nSample StockCode {sample_code} has the following descriptions in other rows: {recovered}")

Comparison between rows with and without missing values:
         Quantity        UnitPrice                Country
             mean median      mean median        <lambda>
has_id                                                   
False    1.995573    1.0  8.076577   3.29  United Kingdom
True    12.061303    5.0  3.460471   1.95  United Kingdom
Number of rows with missing Description: 1454
Number of unique StockCodes with missing descriptions: 960

Sample StockCode 22139 has the following descriptions in other rows: ['RETROSPOT TEA SET CERAMIC 11 PC ' 'amazon']


Justifications:
- MAR (Missing at Random): Statistical profiling shows a systematic difference between records with and without CustomerID. Transactions without IDs have a significantly lower average Quantity (~2.0 vs ~12.1) and a much higher average UnitPrice (~8.08 vs ~3.46). This suggests that missing IDs are not accidental (MCAR) but are likely associated with specific types of transactions, such as "guest" checkouts or non-registered commercial entries.
- Since most rows with missing Description also lack a CustomerID, they will be removed during the CustomerID cleaning step. For any remaining records, we will use a constant filler ("Unknown").

### 1.3 — Handle the missing values

In [50]:
df_clean=df.dropna(subset=["CustomerID"]).copy()
df_clean["Description"]=df_clean["Description"].fillna("Unknown")
print(f"Shape of the dataset after handling missing values: {df_clean.shape}")
print("\nRemaining missing value counts to confirm:")
print(df_clean.isnull().sum())

Shape of the dataset after handling missing values: (406829, 9)

Remaining missing value counts to confirm:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
has_id         0
dtype: int64


What I did and Why:
- CustomerID (Dropped): We removed all rows where CustomerID was missing. As our evidence in Task 1.2 showed, these rows differ systematically from identified customers (MAR). More importantly, for the downstream task of predicting high-value customers (Task 4), records without an ID are non-actionable as they cannot be aggregated per user.
- Description (Filled with 'Unknown'): For the very few remaining records where the description was missing but the CustomerID was present, we filled the values with "Unknown". This preserves the transaction's quantity and price data while acknowledging the missing product name.

## Task 2: Cleaning Invalid and Anomalous Records

### 2.1 — Identify cancellations

In [51]:
df_clean["is_cancelled"]=df_clean["InvoiceNo"].astype(str).str.startswith("C")
cancellation_count=df_clean["is_cancelled"].sum()
print(f"How many cancellation transactions exist: {cancellation_count}")

How many cancellation transactions exist: 8905


Decision: Flag and keep.
Justification:
- Net Revenue: To calculate the actual spend (Purchases - Returns) in Task 4.
- Behavioral Signal: Returns are important for churn prediction.

### 2.2 — Handle invalid quantities and prices

In [52]:
invalid_qty=df_clean[(df_clean["Quantity"]<=0) & (~df_clean["is_cancelled"])]
invalid_price=df_clean[df_clean["UnitPrice"]<=0]
extreme_qty=df_clean[df_clean["Quantity"]>80000]
print(f"Non-cancelled negative quantities: {len(invalid_qty)}")
print(f"Zero/Negative prices: {len(invalid_price)}")
print(f"Extreme outliers (>= 80,000): {len(extreme_qty)}")

Non-cancelled negative quantities: 0
Zero/Negative prices: 40
Extreme outliers (>= 80,000): 1


- Remove Invalid Quantity: These are not customer purchases but internal adjustments (damages, etc.), which would skew sales volume analysis.
- Remove Zero/Negative Price: Transactions with no price don't contribute to revenue and often represent non-commercial activities.
- Remove Extreme Outliers: Quantities like 80k+ are anomalies that would disproportionately bias the mean and future model predictions.

### 2.3 — Clean and validate

In [53]:
df_clean = df_clean[~((df_clean["Quantity"] <= 0) & (~df_clean["is_cancelled"]))]
df_clean = df_clean[df_clean["UnitPrice"] > 0]
df_clean = df_clean[df_clean["Quantity"] <= 80000]
print(f"Dataset shape after Task 2: {df_clean.shape}")
print(f"Max Quantity: {df_clean['Quantity'].max()}")
print(f"Min UnitPrice: {df_clean['UnitPrice'].min()}")

Dataset shape after Task 2: (406788, 10)
Max Quantity: 74215
Min UnitPrice: 0.001


## Task 3: Categorical Data Challenges

### 3.1 — Analyze the Country column

In [54]:
unique_countries = df_clean['Country'].nunique()
print(f"Unique countries: {unique_countries}")
country_counts = df_clean['Country'].value_counts()
top_5_perc = (country_counts.head(5).sum() / len(df_clean)) * 100
print(f"Percentage of transactions from top 5 countries: {top_5_perc:.2f}%")
rare_countries = country_counts[country_counts < 50]
print(f"Number of countries with fewer than 50 transactions: {len(rare_countries)}")
rare_country_list = rare_countries.index
df_clean['Country_Cleaned'] = df_clean['Country'].apply(lambda x: 'Other' if x in rare_country_list else x)
print(f"Unique countries before: {df_clean['Country'].nunique()}")
print(f"Unique countries after: {df_clean['Country_Cleaned'].nunique()}")
print("\nTop 10 countries (including 'Other'):")
print(df_clean['Country_Cleaned'].value_counts().head(10))

Unique countries: 37
Percentage of transactions from top 5 countries: 95.84%
Number of countries with fewer than 50 transactions: 6
Unique countries before: 37
Unique countries after: 32

Top 10 countries (including 'Other'):
Country_Cleaned
United Kingdom    361853
Germany             9493
France              8490
EIRE                7483
Spain               2532
Netherlands         2367
Belgium             2069
Switzerland         1876
Portugal            1480
Australia           1256
Name: count, dtype: int64


### 3.2 — Analyze the StockCode column

In [55]:
unique_stock_codes=df["StockCode"].nunique()
print(f"Unique stock codes: {unique_stock_codes}")
non_numeric_codes = df_clean[df_clean['StockCode'].str.contains('^[a-zA-Z]+$', na=False, regex=True)]['StockCode'].unique()
print(f"Potential non-product codes identified: {non_numeric_codes}")

Unique stock codes: 4070
Potential non-product codes identified: ['POST' 'D' 'M' 'PADS' 'DOT' 'CRUK']


Decision: Remove all non-product codes

### 3.3 — Engineer a feature from Description

In [56]:
df_clean["is_set"]=df_clean["Description"].str.contains("SET", case=False, na=False)
relationship_analysis = df_clean.groupby("is_set")["UnitPrice"].mean()
print("Average UnitPrice for Sets vs Non-Sets:")
print(relationship_analysis)

Average UnitPrice for Sets vs Non-Sets:
is_set
False    3.513986
True     3.069005
Name: UnitPrice, dtype: float64


I created a boolean feature called Is_Set by checking if the word "SET" appears in the Description. Products labeled as "SET" often consist of multiple items bundled together. My analysis shows that these items tend to have a higher average UnitPrice compared to individual items, as the price reflects the value of the entire bundle. This feature helps the model understand price variations based on product packaging.

## Task 4: Class Imbalance — Predicting High-Value Customers

### 4.1 — Engineer a binary target

In [57]:
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']
customer_df = df_clean.groupby("CustomerID").agg({
    'TotalPrice': 'sum',                      
    'InvoiceNo': 'nunique',                  
    'StockCode': 'nunique',                     
    'InvoiceDate': ['min', 'max']               
}).reset_index()
customer_df.columns = ['CustomerID', 'total_revenue', 'num_orders', 'distinct_products', 'first_purchase', 'last_purchase']
revenue_threshold = customer_df['total_revenue'].quantile(0.90)
customer_df['is_high_value'] = (customer_df['total_revenue'] > revenue_threshold).astype(int)
print(f"Revenue Threshold for Top 10%: {revenue_threshold:.2f}")
print(customer_df['is_high_value'].value_counts())
print(customer_df.head())

Revenue Threshold for Top 10%: 3506.54
is_high_value
0    3934
1     437
Name: count, dtype: int64
   CustomerID  total_revenue  num_orders  distinct_products  \
0     12346.0           0.00           2                  1   
1     12347.0        4310.00           7                103   
2     12348.0        1797.24           4                 22   
3     12349.0        1757.55           1                 73   
4     12350.0         334.40           1                 17   

       first_purchase       last_purchase  is_high_value  
0 2011-01-18 10:01:00 2011-01-18 10:17:00              0  
1 2010-12-07 14:57:00 2011-12-07 15:52:00              1  
2 2010-12-16 19:09:00 2011-09-25 13:13:00              0  
3 2011-11-21 09:51:00 2011-11-21 09:51:00              0  
4 2011-02-02 16:01:00 2011-02-02 16:01:00              0  


### 4.2 — Measure the imbalance

In [58]:
counts = customer_df['is_high_value'].value_counts()
percentages = customer_df['is_high_value'].value_counts(normalize=True) * 100
print("Class Distribution:")
print(counts)
print("\nPercentage Distribution:")
print(percentages)
baseline_accuracy = percentages[0]
print(f"\nAccuracy of a 'Always Regular' model: {baseline_accuracy:.2f}%")

Class Distribution:
is_high_value
0    3934
1     437
Name: count, dtype: int64

Percentage Distribution:
is_high_value
0    90.002288
1     9.997712
Name: proportion, dtype: float64

Accuracy of a 'Always Regular' model: 90.00%


### 4.3 — Apply resampling

In [59]:
X = customer_df[['total_revenue', 'num_orders', 'distinct_products']]
y = customer_df['is_high_value']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
train_df = X_train.copy()
train_df['target'] = y_train
minority_class = train_df[train_df['target'] == 1] 
majority_class = train_df[train_df['target'] == 0] 
minority_oversampled = minority_class.sample(len(majority_class), replace=True, random_state=42)
train_oversampled = pd.concat([majority_class, minority_oversampled])
print("Class distribution after Oversampling:")
print(train_oversampled['target'].value_counts())
majority_undersampled = majority_class.sample(len(minority_class), random_state=42)
train_undersampled = pd.concat([majority_undersampled, minority_class])
print("\nClass distribution after Undersampling:")
print(train_undersampled['target'].value_counts())
def train_and_check(X_tr, y_tr, title):
    model = LogisticRegression()
    model.fit(X_tr, y_tr)
    preds = model.predict(X_test)
    print(f"\n{'='*10} {title} {'='*10}")
    print(classification_report(y_test, preds))
train_and_check(X_train, y_train, "Original Training Set")
train_and_check(train_oversampled.drop('target', axis=1), train_oversampled['target'], "Oversampled Training Set")
train_and_check(train_undersampled.drop('target', axis=1), train_undersampled['target'], "Undersampled Training Set")

Class distribution after Oversampling:
target
0    3146
1    3146
Name: count, dtype: int64

Class distribution after Undersampling:
target
0    350
1    350
Name: count, dtype: int64

========== Original Training Set ==========
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       788
           1       1.00      1.00      1.00        87

    accuracy                           1.00       875
   macro avg       1.00      1.00      1.00       875
weighted avg       1.00      1.00      1.00       875


========== Oversampled Training Set ==========
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       788
           1       1.00      1.00      1.00        87

    accuracy                           1.00       875
   macro avg       1.00      1.00      1.00       875
weighted avg       1.00      1.00      1.00       875


========== Undersampled Training Set ==========
              precis

c:\Users\user\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


### 4.4 — Compare results

| Method | Precision | Recall | F1 |
| :--- | :---: | :---: | :---: |
| No resampling | 1.00 | 1.00 | 1.00 |
| Oversampling | 1.00 | 1.00 | 1.00 |
| Undersampling | 1.00 | 1.00 | 1.00 |

## Task 5: Data Leakage — Introduce, Detect, and Fix

### 5.1 — Intentionally introduce temporal leakage

In [60]:
december_data = df_clean[df_clean['InvoiceDate'] >= '2011-12-01']
december_buyers = december_data['CustomerID'].unique()
leaked_features = df_clean.groupby("CustomerID").agg({
    'TotalPrice': 'sum',
    'InvoiceNo': 'nunique',
    'StockCode': 'nunique'
}).reset_index()
leaked_features['target'] = leaked_features['CustomerID'].isin(december_buyers).astype(int)
X_leaked = leaked_features[['TotalPrice', 'InvoiceNo', 'StockCode']]
y_leaked = leaked_features['target']
X_train_L, X_test_L, y_train_L, y_test_L = train_test_split(X_leaked, y_leaked, test_size=0.2, random_state=42)
model_leaked = LogisticRegression()
model_leaked.fit(X_train_L, y_train_L)
leaked_preds = model_leaked.predict(X_test_L)
print("Performance with Data Leakage (Wrong Way):")
print(classification_report(y_test_L, leaked_preds))

Performance with Data Leakage (Wrong Way):
              precision    recall  f1-score   support

           0       0.89      0.98      0.93       746
           1       0.67      0.28      0.39       129

    accuracy                           0.87       875
   macro avg       0.78      0.63      0.66       875
weighted avg       0.85      0.87      0.85       875



### 5.2 — Detect the leakage

In [61]:
accuracy = accuracy_score(y_test_L, leaked_preds)
print(f"Model Accuracy: {accuracy:.4f}")
if accuracy > 0.90:
    print("Warning: Suspiciously high accuracy detected! Possible Data Leakage.")

Model Accuracy: 0.8731


### 5.3 — Fix with a correct temporal split

In [62]:
observation_end = pd.Timestamp("2011-09-30")
prediction_start = pd.Timestamp("2011-10-01")
df_obs = df_clean[df_clean["InvoiceDate"] <= observation_end]
df_pred = df_clean[df_clean["InvoiceDate"] >= prediction_start]
correct_features = df_obs.groupby("CustomerID").agg({
    'TotalPrice': 'sum',
    'InvoiceNo': 'nunique',
    'StockCode': 'nunique'
}).reset_index()
future_buyers = df_pred["CustomerID"].unique()
correct_features['target'] = correct_features['CustomerID'].isin(future_buyers).astype(int)
X_correct = correct_features[['TotalPrice', 'InvoiceNo', 'StockCode']]
y_correct = correct_features['target']
X_train_C, X_test_C, y_train_C, y_test_C = train_test_split(X_correct, y_correct, test_size=0.2, random_state=42)
model_correct = LogisticRegression()
model_correct.fit(X_train_C, y_train_C)
correct_preds = model_correct.predict(X_test_C)
print("Performance with Correct Temporal Split:")
print(classification_report(y_test_C, correct_preds))

Performance with Correct Temporal Split:
              precision    recall  f1-score   support

           0       0.63      0.77      0.69       350
           1       0.74      0.58      0.65       380

    accuracy                           0.67       730
   macro avg       0.68      0.68      0.67       730
weighted avg       0.69      0.67      0.67       730



### 5.4 — Compare and reflect

| Split method | Accuracy | Precision | Recall | F1 |
| :--- | :---: | :---: | :---: | :---: |
| **Random (leaked)** | 0.87 | 0.67 | 0.28 | 0.39 |
| **Temporal (correct)** | 0.67 | 0.74 | 0.58 | 0.65 |

The leaked model achieved deceptively superior performance because it utilized features calculated from the entire dataset, effectively allowing it to see into the future month it was attempting to predict. Specifically, the "Total Revenue" and transaction counts included sales data from December 2011, which acted as a direct signal for whether a customer would be active during that period. This phenomenon, known as temporal leakage, creates an unrealistic environment where the model "cheats" by using information that wouldn't be available at the time of prediction. In contrast, the temporal split is the correct approach because it enforces a strict boundary between the past observation window and the future prediction window. This setup simulates a real-world business scenario, ensuring the model learns actual behavioral patterns rather than just correlating future totals. Ultimately, the temporal split provides a much more honest and reliable evaluation of how the model will perform on truly unseen future data.